In [46]:
import colorsys
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, ColorPicker
import matplotlib.patches as patches

In [47]:
def rgb_to_hex(rgb):
    return "#{:02X}{:02X}{:02X}".format(*rgb)

In [48]:
# Função para gerar harmonias
def gerar_harmonia_hsv(base_color=(120, 0, 20), tipo='monochromatic', n=5):
    """
    Gera uma paleta de cores a partir de uma cor base em HSV/HSB.
    Para monocromático, ajusta matiz, saturação e brilho para parecer mais distinto.
    """
    r, g, b = [x/255 for x in base_color]
    h_base, s_base, v_base = colorsys.rgb_to_hsv(r, g, b)
    cores = []

    if tipo == 'monochromatic':
        # Monocromático mais perceptível: variação de V, S e leve H
        for i in range(n):
            # Variação do brilho com curva não-linear
            fator_v = 0.4 + (i/(n-1))**1.5 * 0.6  # entre 0.4 e 1
            v_i = v_base * fator_v
            # Variação sutil da saturação
            s_i = max(min(s_base * (0.7 + 0.3 * (i/(n-1))), 1), 0)
            # Pequena variação de matiz ±3° (~0.008 no 0-1)
            h_i = (h_base + (i - n//2) * 0.008) % 1
            r2, g2, b2 = colorsys.hsv_to_rgb(h_i, s_i, v_i)
            cores.append(tuple(round(c*255) for c in (r2, g2, b2)))

    elif tipo == 'analogous':
        shift = 1/24
        for i in range(-n//2, n//2 + 1):
            h_i = (h_base + i * shift) % 1
            s_i = max(s_base, 0.6)
            v_i = max(v_base, 0.5)
            r2, g2, b2 = colorsys.hsv_to_rgb(h_i, s_i, v_i)
            cores.append(tuple(round(c*255) for c in (r2, g2, b2)))

    elif tipo == 'complementary':
        s_adj = max(s_base, 0.7)
        v_adj = max(v_base, 0.5)
        cores = [
            tuple(round(c*255) for c in colorsys.hsv_to_rgb(h_base, s_base, v_base)),
            tuple(round(c*255) for c in colorsys.hsv_to_rgb((h_base + 0.5) % 1, s_adj, v_adj))
        ]

    elif tipo == 'triadic':
        for i in range(3):
            h_i = (h_base + i/3) % 1
            s_i = max(s_base, 0.7)
            v_i = max(v_base, 0.5)
            r2, g2, b2 = colorsys.hsv_to_rgb(h_i, s_i, v_i)
            cores.append(tuple(round(c*255) for c in (r2, g2, b2)))

    elif tipo == 'square':
        for i in range(4):
            h_i = (h_base + i * 0.25) % 1
            s_i = max(s_base, 0.7)
            v_i = max(v_base, 0.5)
            r2, g2, b2 = colorsys.hsv_to_rgb(h_i, s_i, v_i)
            cores.append(tuple(round(c*255) for c in (r2, g2, b2)))

    else:
        raise ValueError("Tipo inválido")
    
    return cores

In [49]:
# # Função de plotagem interativa compatível com slider
# def plotar_paleta_hsv(base_hex='#780014', tipo='monochromatic', n=5):
#     # Converter HEX para RGB
#     base_color = tuple(int(base_hex[i:i+2], 16) for i in (1, 3, 5))
#     paleta = gerar_harmonia_hsv(base_color, tipo=tipo, n=n)
#     paleta_norm = [(r/255, g/255, b/255) for r, g, b in paleta]
    
#     fig, ax = plt.subplots(figsize=(len(paleta)*1.2, 2))
#     for i, cor in enumerate(paleta_norm):
#         # Desenhar retângulo da cor
#         ax.add_patch(patches.Rectangle((i, 0), 1, 1, color=cor))
#         # Mostrar RGB e HEX dentro do retângulo
#         hex_cor = rgb_to_hex(paleta[i])
#         ax.text(i + 0.5, 0.5, f"{hex_cor}", ha='center', va='center', fontsize=10, color='white' if sum(cor) < 1.5 else 'black', weight='bold')
    
#     ax.set_xlim(0, len(paleta))
#     ax.set_ylim(0, 1)
#     ax.axis('off')
#     plt.show()
def plotar_paleta_hsv(base_hex='#780014', tipo='monochromatic', n=5):
    # Converter HEX para RGB
    base_color = tuple(int(base_hex[i:i+2], 16) for i in (1, 3, 5))
    
    # Definir número fixo de cores por esquema
    if tipo == 'complementary':
        n_cores = 2
    elif tipo == 'triadic':
        n_cores = 3
    elif tipo == 'square':
        n_cores = 4
    else:  # monochromatic ou outro tipo
        n_cores = n
    
    paleta = gerar_harmonia_hsv(base_color, tipo=tipo, n=n_cores)
    paleta_norm = [(r/255, g/255, b/255) for r, g, b in paleta]

    spacing = 0.05  # Espaço entre os retângulos
    width = 1 - spacing

    fig, ax = plt.subplots(figsize=(len(paleta)*1.3, 2))
    for i, cor in enumerate(paleta_norm):
        x = i * (1 + spacing)
        ax.add_patch(patches.Rectangle((x, 0), width, 1, color=cor))
        hex_cor = rgb_to_hex(paleta[i])
        cor_texto = 'white' if sum(cor) < 1.5 else 'black'
        ax.text(x + width/2, 0.5, hex_cor, ha='center', va='center', fontsize=10, color=cor_texto, weight='bold')

    ax.set_xlim(0, n_cores * (1 + spacing) - spacing)
    ax.set_ylim(0, 1)
    ax.axis('off')
    plt.show()


In [ ]:
interact(
    plotar_paleta_hsv,
    base_hex=ColorPicker(value='#780014', description='Cor Base'),
    tipo=Dropdown(
        options=['monochromatic','analogous','complementary','square','triadic'],
        value='monochromatic',
        description='Harmonia'
    ),
    n=IntSlider(min=2, max=15, step=1, value=5, description='Cores')
)

interactive(children=(ColorPicker(value='#780014', description='Cor Base'), Dropdown(description='Harmonia', o…

<function __main__.plotar_paleta_hsv(base_hex='#780014', tipo='monochromatic', n=5)>